In [13]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()
current_dir = Path.cwd()
project_root = current_dir.resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Added to path: {project_root}")

Added to path: /users/oshan/Dev/financial-document-based-agent-system


In [14]:
from src.dochandler.main import ExTrRAGDocHandler
from src.cgcore.vectordb.milvus import MilvusDB
from src.cgcore.embedder.openai import OpenAIEmbedder
from src.cgcore.llm.openai import OpenAILlm

from src.cgcore.configs.vectordb.milvus import MilvusConfig
from src.cgcore.configs.embedder.openai import OpenAIEmbedderConfig
from src.cgcore.configs.llm.openai import OpenAILlmConfig

In [15]:
llm_config = OpenAILlmConfig(api_key=os.getenv('OPENAI_API_KEY'))
embedder_config = OpenAIEmbedderConfig(api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
vectordb_config = MilvusConfig(
                collection_name=os.getenv('MILVUS_COLLECTION_NAME'),
                dimensions=1536,  # Set explicit dimension value
                )


In [16]:
vectordb_config.collection_name

'financial_documents_backend_test222'

In [17]:
openai = OpenAILlm(llm_config)
embeder = OpenAIEmbedder(embedder_config)
vectordb = MilvusDB(vectordb_config)

AsyncMilvusClient initialized.


/tmp/ipykernel_3230681/1873515904.py:1: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  openai = OpenAILlm(llm_config)


## ExTr RAG 

In [18]:
extr_rag = ExTrRAGDocHandler(
    llm=openai,
    embedder=embeder,
    db=vectordb,
    memory="none",
    history=True
)

## Document Loading test


In [19]:
extr_rag.loader.dry_run = False

In [20]:
pdf_path = "/users/oshan/Dev/financial-document-based-agent-system/FIU_AR_2011.pdf"

In [21]:
print(f"Testing Processing for: {os.path.basename(pdf_path)} ---")

Testing Processing for: FIU_AR_2011.pdf ---


In [22]:
# from dochandler.src.loader.dockling_loader import DocklingLoader
# from dochandler.src.chunkers.mdx import MdxChunker

# loader = DocklingLoader(server_url="http://localhost:8080/documents/convert")
# # creates the .md file locally
# raw_data = loader.load_data(pdf_path) 
# print(f"Conversion complete. Markdown size: {len(raw_data['data'][0]['content'])} chars")

# chunker = MdxChunker() 
# chunks = chunker.create_chunks(loader, pdf_path)

# print(f"\n HUNKING RESULTS:")
# print(f"Total Chunks Created: {len(chunks['ids'])}")


In [23]:
# for i in range(min(30, len(chunks['ids']))):
#     chunk_id = chunks['ids'][i]
#     content = chunks['documents'][i]
#     meta = chunks['metadatas'][i]
    
#     print(f"\n CHUNK #{i} [ID: {chunk_id}]")
#     print(f"   Metadata: {meta}")
#     print(f"   {content.strip()[:300]}...") 



### Questions generation and saving in the VDB

In [24]:
records = await extr_rag.add_document(pdf_path)

Loading 'FIU_AR_2011.pdf' via Dockling Server...


ic| questions: ['What is the main highlight of the Annual Report 2011?',
                'Can you provide a summary of the financial performance in the Annual Report '
                '2011?',
                'What are the key achievements mentioned in the Annual Report 2011?']
ic| questions: ['What is the name of the organization mentioned in the image?',
                'What year is the Annual Report from?',
                'Which institution is the Financial Intelligence Unit of Sri Lanka a part of?']
ic| questions: ['What is the ISBN of the book?',
                'Who is the printer of the book?',
                'Where is the printer located?',
                'Who is the publisher of the book?',
                'Where is the publisher located?']
ic| questions: ['What is the main concern of the Central Bank of Sri Lanka mentioned in the '
                'message?',
                'What laws were enacted in 2005 and 2006 to address money laundering and '
                'terror

Collection 'financial_documents_backend_test222' created successfully.
Successfully inserted 91 records into Milvus.
